In [1]:
# =============================================================================
# 12_create_biweekly_satellite_data.ipynb
#
# PURPOSE
# =============================================================================
#
# Create 14-day Sentinel-1 and Sentinel-2 composite imagery from the
# acquisition-level satellite dataset created by Notebook 10.
#
#
# SOURCE DATA
# =============================================================================
#
# finals/
# └── daily_datasets/
#
#     ├── sentinel1/
#     ├── sentinel2/
#     ├── selected_site_sample.csv
#     └── daily_satellite_inventory.csv
#
#
# SAMPLE
# =============================================================================
#
# Notebook 10 defines the sample.
#
# Expected:
#
#   10 treatment sites
#
#   5 matched counterfactual sites per treatment
#
#   50 counterfactual sites
#
#   60 total sites
#
#
# Notebook 12 DOES NOT independently choose treatment/control sites.
#
# selected_site_sample.csv defines the fixed spatial panel.
#
#
# IMPORTANT
# =============================================================================
#
# Notebook 12 DOES NOT:
#
#   - connect to Earth Engine
#   - download satellite imagery
#   - modify daily/acquisition-level imagery
#   - redefine treatment/control matching
#   - create another daily dataset
#
#
# TEMPORAL DESIGN
# =============================================================================
#
# BEFORE:
#
#   2024-05-10 through 2024-09-26
#
#   140 calendar days
#
#   exactly:
#
#       before_P01
#       ...
#       before_P10
#
#
# AFTER:
#
#   2024-09-27 through 2025-02-13
#
#   140 calendar days
#
#   exactly:
#
#       after_P01
#       ...
#       after_P10
#
#
# Each period = exactly 14 calendar days.
#
# September 27, 2024 is the first AFTER date.
#
# No biweekly period crosses the hurricane cutoff.
#
#
# COMPOSITING RULE
# =============================================================================
#
# For each:
#
#   site × sensor × 14-day period
#
#
# 0 acquisitions:
#
#   - no TIFF created
#   - row retained in panel
#   - has_data = 0
#   - quality = NaN
#
#
# 1 acquisition:
#
#   - composite equals that acquisition
#
#
# 2+ acquisitions:
#
#   - pixel-wise NaN-aware temporal median
#
#
# Example:
#
# Acquisition 1 pixel = NaN
# Acquisition 2 pixel = 0.52
# Acquisition 3 pixel = 0.61
#
# Composite pixel:
#
#   median(0.52, 0.61)
#
#
# PRIMARY QUALITY DEFINITION
# =============================================================================
#
# A spatial pixel is VALID when AT LEAST ONE output band contains
# a finite value.
#
#
# Primary:
#
#   valid_pixel_fraction
#   valid_pixel_percentage
#
#
# Diagnostic:
#
#   valid_pixel_fraction_any_band
#   valid_pixel_fraction_all_bands
#
#
# RESUME-SAFE BEHAVIOR
# =============================================================================
#
# Existing valid biweekly TIFF:
#
#   - do not rebuild
#   - calculate quality locally
#   - build_action = skipped_existing
#
#
# Existing invalid TIFF:
#
#   - rename to *.invalid
#   - rebuild from daily TIFFs
#
#
# Missing TIFF:
#
#   - build from daily TIFFs
#
#
# FIXED PANEL
# =============================================================================
#
# Expected panel:
#
#   60 sites
#   × 2 sensors
#   × 20 periods
#
#   = 2,400 site × sensor × period rows
#
#
# Missing satellite observations remain explicit missing rows.
#
#
# OUTPUT
# =============================================================================
#
# finals/
# └── biweekly_datasets/
#
#     ├── sentinel1/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── sentinel2/
#     │   ├── treatment/
#     │   │   ├── before/
#     │   │   └── after/
#     │   └── counterfactual/
#     │       ├── before/
#     │       └── after/
#     │
#     ├── biweekly_image_quality.csv
#     ├── biweekly_image_quality.xlsx
#     ├── biweekly_quality_summary.csv
#     ├── biweekly_period_definitions.csv
#     ├── biweekly_period_dimension.csv
#     ├── biweekly_site_dimension.csv
#     ├── biweekly_build_action_summary.csv
#     ├── biweekly_folder_summary.csv
#     └── biweekly_sample_validation.csv
#
# =============================================================================


# =============================================================================
# 1. Packages
# =============================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import rasterio


warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)


print(
    "Packages loaded successfully."
)


# =============================================================================
# 2. Project paths
# =============================================================================

BASE_DIR = Path(
    "/Users/gaoyujuan/REAP Dropbox/Gao yujuan/"
    "Virginia Tech/CALS/datasets"
)


FINALS_DIR = (
    BASE_DIR /
    "finals"
)


# -----------------------------------------------------------------------------
# Notebook 10 source
# -----------------------------------------------------------------------------

DAILY_DIR = (
    FINALS_DIR /
    "daily_datasets"
)


DAILY_INVENTORY_FILE = (
    DAILY_DIR /
    "daily_satellite_inventory.csv"
)


SELECTED_SAMPLE_FILE = (
    DAILY_DIR /
    "selected_site_sample.csv"
)


# -----------------------------------------------------------------------------
# Notebook 12 output
# -----------------------------------------------------------------------------

BIWEEKLY_DIR = (
    FINALS_DIR /
    "biweekly_datasets"
)


S1_BIWEEKLY_DIR = (
    BIWEEKLY_DIR /
    "sentinel1"
)


S2_BIWEEKLY_DIR = (
    BIWEEKLY_DIR /
    "sentinel2"
)


QUALITY_CSV_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.csv"
)


QUALITY_EXCEL_FILE = (
    BIWEEKLY_DIR /
    "biweekly_image_quality.xlsx"
)


SUMMARY_CSV_FILE = (
    BIWEEKLY_DIR /
    "biweekly_quality_summary.csv"
)


PERIOD_DEFINITION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_period_definitions.csv"
)


PERIOD_DIMENSION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_period_dimension.csv"
)


SITE_DIMENSION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_site_dimension.csv"
)


BUILD_ACTION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_build_action_summary.csv"
)


FOLDER_SUMMARY_FILE = (
    BIWEEKLY_DIR /
    "biweekly_folder_summary.csv"
)


SAMPLE_VALIDATION_FILE = (
    BIWEEKLY_DIR /
    "biweekly_sample_validation.csv"
)


BIWEEKLY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 3. Check Notebook 10 outputs
# =============================================================================

required_files = [

    DAILY_INVENTORY_FILE,

    SELECTED_SAMPLE_FILE,

]


missing_files = [

    file_path

    for file_path in required_files

    if not file_path.exists()

]


if missing_files:

    raise FileNotFoundError(
        "Required Notebook 10 outputs are missing:\n\n"
        +
        "\n".join(
            str(file_path)
            for file_path in missing_files
        )
        +
        "\n\nRun Notebook 10 first."
    )


print(
    "\nNotebook 10 daily inventory:"
)


print(
    DAILY_INVENTORY_FILE
)


print(
    "\nNotebook 10 selected sample:"
)


print(
    SELECTED_SAMPLE_FILE
)


# =============================================================================
# 4. Study calendar
# =============================================================================

HELENE_REFERENCE_DATE = pd.Timestamp(
    "2024-09-27"
)


BEFORE_START = pd.Timestamp(
    "2024-05-10"
)


BEFORE_END = pd.Timestamp(
    "2024-09-26"
)


AFTER_START = pd.Timestamp(
    "2024-09-27"
)


AFTER_END = pd.Timestamp(
    "2025-02-13"
)


STUDY_START = (
    BEFORE_START
)


STUDY_END = (
    AFTER_END
)


print(
    "\nStudy period:"
)


print(
    STUDY_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    STUDY_END.strftime(
        "%Y-%m-%d"
    ),
)


print(
    "\nBefore:"
)


print(
    BEFORE_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    BEFORE_END.strftime(
        "%Y-%m-%d"
    ),
)


print(
    "\nAfter:"
)


print(
    AFTER_START.strftime(
        "%Y-%m-%d"
    ),
    "through",
    AFTER_END.strftime(
        "%Y-%m-%d"
    ),
)


# =============================================================================
# 5. Settings
# =============================================================================

# Reuse already-created valid biweekly TIFFs.
SKIP_EXISTING_BIWEEKLY = True


# False = process complete selected sample.
RUN_TEST_ONLY = False


# =============================================================================
# 6. Expected sample structure
# =============================================================================

EXPECTED_TREATMENT_SITES = 10


EXPECTED_CONTROLS_PER_TREATMENT = 5


EXPECTED_COUNTERFACTUAL_SITES = (
    EXPECTED_TREATMENT_SITES
    *
    EXPECTED_CONTROLS_PER_TREATMENT
)


EXPECTED_TOTAL_SITES = (
    EXPECTED_TREATMENT_SITES
    +
    EXPECTED_COUNTERFACTUAL_SITES
)


EXPECTED_PERIODS_BEFORE = 10


EXPECTED_PERIODS_AFTER = 10


EXPECTED_PERIODS_TOTAL = (
    EXPECTED_PERIODS_BEFORE
    +
    EXPECTED_PERIODS_AFTER
)


# =============================================================================
# 7. Sensor bands
# =============================================================================

SENSOR_BANDS = {

    "sentinel1": [

        "VV",

        "VH",

        "VV_minus_VH",

    ],


    "sentinel2": [

        "B2",

        "B3",

        "B4",

        "B8",

        "B11",

        "B12",

        "NDVI",

        "NDWI",

    ],

}


# =============================================================================
# 8. Create output folders
# =============================================================================

for sensor_root in [

    S1_BIWEEKLY_DIR,

    S2_BIWEEKLY_DIR,

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (

                sensor_root /
                group /
                period

            )


            folder.mkdir(
                parents=True,
                exist_ok=True,
            )


print(
    "\nBiweekly output directory:"
)


print(
    BIWEEKLY_DIR
)


# =============================================================================
# 9. Load selected Notebook 10 sample
# =============================================================================

selected_sample = pd.read_csv(
    SELECTED_SAMPLE_FILE
)


required_sample_columns = [

    "site_id",

    "group",

    "matched_treatment_site_id",

    "control_rank",

]


missing_sample_columns = [

    column

    for column in required_sample_columns

    if column not in selected_sample.columns

]


if missing_sample_columns:

    raise ValueError(
        "selected_site_sample.csv is missing columns:\n"
        +
        str(
            missing_sample_columns
        )
    )


selected_sample[
    "site_id"
] = (
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


selected_sample[
    "group"
] = (
    selected_sample[
        "group"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


selected_sample[
    "matched_treatment_site_id"
] = (
    selected_sample[
        "matched_treatment_site_id"
    ]
    .astype("string")
)


selected_sample[
    "control_rank"
] = pd.to_numeric(
    selected_sample[
        "control_rank"
    ],
    errors=
        "coerce",
)


# =============================================================================
# 10. Ensure treatment rows map to themselves
# =============================================================================

treatment_mask = (
    selected_sample[
        "group"
    ]
    ==
    "treatment"
)


selected_sample.loc[
    treatment_mask,
    "matched_treatment_site_id",
] = (
    selected_sample.loc[
        treatment_mask,
        "site_id",
    ]
    .astype(str)
)


# =============================================================================
# 11. Remove accidental duplicate spatial records
# =============================================================================

duplicate_sites = (
    selected_sample
    .duplicated(
        subset=[
            "site_id",
            "group",
        ],
        keep=False,
    )
)


if duplicate_sites.any():

    print(
        "\nWARNING:"
    )


    print(
        "Duplicate site records were found in selected_site_sample.csv."
    )


    print(
        "Duplicate records will be collapsed."
    )


selected_sample = (
    selected_sample
    .drop_duplicates(
        subset=[
            "site_id",
            "group",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 12. Validate selected sample
# =============================================================================

treatment_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "treatment"
    ]
    .copy()
)


counterfactual_sample = (
    selected_sample
    .loc[
        selected_sample[
            "group"
        ]
        ==
        "counterfactual"
    ]
    .copy()
)


treatment_count = (
    treatment_sample[
        "site_id"
    ]
    .nunique()
)


counterfactual_count = (
    counterfactual_sample[
        "site_id"
    ]
    .nunique()
)


total_site_count = (
    selected_sample[
        "site_id"
    ]
    .nunique()
)


print(
    "\n"
    + "=" * 100
)


print(
    "SELECTED SAMPLE"
)


print(
    "=" * 100
)


print(
    "\nTreatment sites:",
    treatment_count
)


print(
    "Counterfactual sites:",
    counterfactual_count
)


print(
    "Total sites:",
    total_site_count
)


# =============================================================================
# 13. Validate 5 controls per treatment
# =============================================================================

controls_per_treatment = (
    counterfactual_sample
    .groupby(
        "matched_treatment_site_id"
    )[
        "site_id"
    ]
    .nunique()
    .rename(
        "counterfactual_count"
    )
    .reset_index()
)


sample_validation = pd.DataFrame(
    {

        "matched_treatment_site_id":
            treatment_sample[
                "site_id"
            ]
            .drop_duplicates()
            .astype(str)
            .tolist(),

    }
)


sample_validation = (
    sample_validation
    .merge(
        controls_per_treatment,
        on=
            "matched_treatment_site_id",
        how=
            "left",
    )
)


sample_validation[
    "counterfactual_count"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    .fillna(
        0
    )
    .astype(int)
)


sample_validation[
    "expected_counterfactual_count"
] = (
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation[
    "sample_complete"
] = (
    sample_validation[
        "counterfactual_count"
    ]
    ==
    EXPECTED_CONTROLS_PER_TREATMENT
)


sample_validation.to_csv(
    SAMPLE_VALIDATION_FILE,
    index=False,
)


print(
    "\nControls per treatment:"
)


print(
    sample_validation.to_string(
        index=False
    )
)


# =============================================================================
# 14. Sample warnings
# =============================================================================

if treatment_count != EXPECTED_TREATMENT_SITES:

    print(
        "\nWARNING:"
    )


    print(
        "Expected",
        EXPECTED_TREATMENT_SITES,
        "treatment sites but found",
        treatment_count,
    )


if counterfactual_count != EXPECTED_COUNTERFACTUAL_SITES:

    print(
        "\nWARNING:"
    )


    print(
        "Expected",
        EXPECTED_COUNTERFACTUAL_SITES,
        "counterfactual sites but found",
        counterfactual_count,
    )


if not sample_validation[
    "sample_complete"
].all():

    print(
        "\nWARNING:"
    )


    print(
        "At least one treatment does not have exactly five controls."
    )


# =============================================================================
# 15. Generate exactly 10 before + 10 after 14-day periods
# =============================================================================

def create_14day_periods(
    period_name,
    start_date,
    number_of_periods=10,
):

    records = []


    for period_number in range(
        1,
        number_of_periods + 1,
    ):

        period_start = (
            start_date
            +
            pd.Timedelta(
                days=
                    (
                        period_number - 1
                    )
                    *
                    14
            )
        )


        period_end = (
            period_start
            +
            pd.Timedelta(
                days=13
            )
        )


        records.append(
            {

                "period":
                    period_name,

                "period_number":
                    period_number,

                "period_id":
                    (
                        f"{period_name}_"
                        f"P{period_number:02d}"
                    ),

                "period_start":
                    period_start,

                "period_end":
                    period_end,

                "calendar_days":
                    14,

            }
        )


    return records


period_definitions = pd.DataFrame(

    create_14day_periods(
        period_name=
            "before",

        start_date=
            BEFORE_START,

        number_of_periods=
            EXPECTED_PERIODS_BEFORE,
    )

    +

    create_14day_periods(
        period_name=
            "after",

        start_date=
            AFTER_START,

        number_of_periods=
            EXPECTED_PERIODS_AFTER,
    )

)


# =============================================================================
# 16. Validate temporal structure
# =============================================================================

before_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "before"
    ]
    .copy()
)


after_definition = (
    period_definitions
    .loc[
        period_definitions[
            "period"
        ]
        ==
        "after"
    ]
    .copy()
)


assert len(
    before_definition
) == EXPECTED_PERIODS_BEFORE


assert len(
    after_definition
) == EXPECTED_PERIODS_AFTER


assert (
    before_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    BEFORE_START
)


assert (
    before_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    BEFORE_END
)


assert (
    after_definition.iloc[
        0
    ][
        "period_start"
    ]
    ==
    AFTER_START
)


assert (
    after_definition.iloc[
        -1
    ][
        "period_end"
    ]
    ==
    AFTER_END
)


# Ensure there is no temporal overlap.
assert (
    before_definition.iloc[
        -1
    ][
        "period_end"
    ]
    <
    after_definition.iloc[
        0
    ][
        "period_start"
    ]
)


period_definitions.to_csv(
    PERIOD_DEFINITION_FILE,
    index=False,
)


print(
    "\nBiweekly period definitions:"
)


print(
    period_definitions[
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 17. Load Notebook 10 daily inventory
# =============================================================================

inventory = pd.read_csv(
    DAILY_INVENTORY_FILE
)


required_columns = [

    "site_id",

    "group",

    "sensor",

    "acquisition_date",

    "file_path",

]


missing_columns = [

    column

    for column in required_columns

    if column not in inventory.columns

]


if missing_columns:

    raise ValueError(
        "Notebook 10 inventory is missing required columns:\n"
        +
        str(
            missing_columns
        )
    )


inventory[
    "site_id"
] = (
    inventory[
        "site_id"
    ]
    .astype(str)
)


inventory[
    "group"
] = (
    inventory[
        "group"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


inventory[
    "sensor"
] = (
    inventory[
        "sensor"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


inventory[
    "acquisition_date"
] = pd.to_datetime(
    inventory[
        "acquisition_date"
    ],
    errors=
        "coerce",
)


inventory = (
    inventory
    .loc[
        inventory[
            "acquisition_date"
        ]
        .notna()
    ]
    .copy()
)


print(
    "\nRaw Notebook 10 inventory rows:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 18. Restrict inventory to fixed selected sample
# =============================================================================

selected_site_ids = set(
    selected_sample[
        "site_id"
    ]
    .astype(str)
)


inventory = (
    inventory
    .loc[
        inventory[
            "site_id"
        ]
        .isin(
            selected_site_ids
        )
    ]
    .copy()
)


# =============================================================================
# 19. Remove duplicate inventory records
#
# A duplicate site × sensor × acquisition_date record should not count twice.
# =============================================================================

inventory = (
    inventory
    .sort_values(
        [
            "site_id",
            "sensor",
            "acquisition_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "site_id",
            "group",
            "sensor",
            "acquisition_date",
            "file_path",
        ],
        keep=
            "first",
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 20. Keep successful / existing acquisition TIFFs
# =============================================================================

if "status" in inventory.columns:

    inventory = (
        inventory
        .loc[
            inventory[
                "status"
            ]
            .isin(
                [
                    "success",
                    "existing",
                ]
            )
        ]
        .copy()
    )


# =============================================================================
# 21. Restrict to exact study period
# =============================================================================

inventory = (
    inventory
    .loc[
        (
            inventory[
                "acquisition_date"
            ]
            >=
            STUDY_START
        )
        &
        (
            inventory[
                "acquisition_date"
            ]
            <=
            STUDY_END
        )
    ]
    .copy()
)


# =============================================================================
# 22. Check local source TIFFs
# =============================================================================

inventory[
    "file_exists"
] = (
    inventory[
        "file_path"
    ]
    .astype(str)
    .apply(
        lambda path:
            Path(
                path
            ).exists()
    )
)


missing_source_files = (
    inventory
    .loc[
        ~inventory[
            "file_exists"
        ]
    ]
    .copy()
)


if not missing_source_files.empty:

    print(
        "\nWARNING:"
    )


    print(
        len(
            missing_source_files
        ),
        "daily inventory rows point to missing TIFFs."
    )


    print(
        "Those acquisitions will not be used."
    )


inventory = (
    inventory
    .loc[
        inventory[
            "file_exists"
        ]
    ]
    .copy()
)


print(
    "\nUsable acquisition-level images:"
)


print(
    len(
        inventory
    )
)


# =============================================================================
# 23. Inventory diagnostics
# =============================================================================

print(
    "\nUsable acquisitions by sensor:"
)


print(
    inventory[
        "sensor"
    ]
    .value_counts()
)


print(
    "\nUsable acquisitions by group:"
)


print(
    inventory[
        "group"
    ]
    .value_counts()
)


# =============================================================================
# 24. Read GeoTIFF
# =============================================================================

def read_raster(
    file_path,
):

    file_path = Path(
        file_path
    )


    with rasterio.open(
        file_path
    ) as src:

        data = (
            src
            .read(
                masked=True
            )
            .astype(
                "float32"
            )
            .filled(
                np.nan
            )
        )


        result = {

            "data":
                data,

            "profile":
                src.profile.copy(),

            "transform":
                src.transform,

            "crs":
                src.crs,

            "width":
                int(
                    src.width
                ),

            "height":
                int(
                    src.height
                ),

            "band_count":
                int(
                    src.count
                ),

        }


    return result


# =============================================================================
# 25. Inspect TIFF and calculate quality
# =============================================================================

def inspect_tiff(
    file_path,
    band_names,
):

    file_path = Path(
        file_path
    )


    if not file_path.exists():

        return {

            "reusable":
                False,

            "validation_status":
                "missing",

            "error":
                "File does not exist.",

        }


    try:

        raster = (
            read_raster(
                file_path
            )
        )


        if (
            raster[
                "band_count"
            ]
            !=
            len(
                band_names
            )
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "wrong_band_count",

                "error":
                    (
                        f"Expected {len(band_names)} bands, "
                        f"found {raster['band_count']}."
                    ),

            }


        if (
            raster[
                "width"
            ]
            <=
            0
            or
            raster[
                "height"
            ]
            <=
            0
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "invalid_dimensions",

                "error":
                    "Raster dimensions are invalid.",

            }


        if (
            raster[
                "crs"
            ]
            is None
        ):

            return {

                "reusable":
                    False,

                "validation_status":
                    "missing_crs",

                "error":
                    "Raster has no CRS.",

            }


        data = (
            raster[
                "data"
            ]
        )


        finite = np.isfinite(
            data
        )


        # ---------------------------------------------------------------------
        # PRIMARY:
        # at least one output band is finite.
        # ---------------------------------------------------------------------

        valid_any = (
            finite
            .any(
                axis=0
            )
        )


        # ---------------------------------------------------------------------
        # Strict diagnostic:
        # all output bands are finite.
        # ---------------------------------------------------------------------

        valid_all = (
            finite
            .all(
                axis=0
            )
        )


        result = {

            "reusable":
                True,

            "validation_status":
                "valid",

            "error":
                None,

            "valid_pixel_fraction":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_any_band":
                float(
                    valid_any.mean()
                ),

            "valid_pixel_percentage_any_band":
                float(
                    valid_any.mean()
                    *
                    100
                ),

            "valid_pixel_fraction_all_bands":
                float(
                    valid_all.mean()
                ),

            "valid_pixel_percentage_all_bands":
                float(
                    valid_all.mean()
                    *
                    100
                ),

        }


        for band_index, band_name in enumerate(
            band_names
        ):

            fraction = float(
                finite[
                    band_index
                ]
                .mean()
            )


            result[
                f"valid_fraction_{band_name}"
            ] = (
                fraction
            )


            result[
                f"valid_percentage_{band_name}"
            ] = (
                fraction
                *
                100
            )


        return result


    except Exception as error:

        return {

            "reusable":
                False,

            "validation_status":
                "corrupt_or_unreadable",

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 26. Raster compatibility
# =============================================================================

def check_raster_compatibility(
    raster_infos,
):

    if len(
        raster_infos
    ) <= 1:

        return (
            True,
            None,
        )


    reference = (
        raster_infos[
            0
        ]
    )


    for image_number, current in enumerate(
        raster_infos[
            1:
        ],
        start=2,
    ):

        # ---------------------------------------------------------------------
        # Shape
        # ---------------------------------------------------------------------

        if (
            current[
                "data"
            ].shape
            !=
            reference[
                "data"
            ].shape
        ):

            return (

                False,

                (
                    f"Image {image_number} shape mismatch: "
                    f"{current['data'].shape} vs "
                    f"{reference['data'].shape}"
                ),

            )


        # ---------------------------------------------------------------------
        # CRS
        # ---------------------------------------------------------------------

        if (
            str(
                current[
                    "crs"
                ]
            )
            !=
            str(
                reference[
                    "crs"
                ]
            )
        ):

            return (

                False,

                f"Image {image_number} CRS mismatch.",

            )


        # ---------------------------------------------------------------------
        # Pixel grid
        # ---------------------------------------------------------------------

        if (
            current[
                "transform"
            ]
            !=
            reference[
                "transform"
            ]
        ):

            return (

                False,

                f"Image {image_number} pixel-grid mismatch.",

            )


    return (
        True,
        None,
    )


# =============================================================================
# 27. Create biweekly composite
# =============================================================================

def create_biweekly_composite(
    source_files,
):

    if len(
        source_files
    ) == 0:

        raise ValueError(
            "No source TIFFs were supplied."
        )


    raster_infos = [

        read_raster(
            file_path
        )

        for file_path in source_files

    ]


    compatible, error = (
        check_raster_compatibility(
            raster_infos
        )
    )


    if not compatible:

        raise ValueError(
            error
        )


    # -------------------------------------------------------------------------
    # One acquisition
    # -------------------------------------------------------------------------

    if len(
        raster_infos
    ) == 1:

        return (

            raster_infos[
                0
            ][
                "data"
            ]
            .copy(),

            raster_infos[
                0
            ],

        )


    # -------------------------------------------------------------------------
    # Multiple acquisitions
    # -------------------------------------------------------------------------

    stack = np.stack(

        [

            raster[
                "data"
            ]

            for raster in raster_infos

        ],

        axis=0,

    )


    # acquisition × band × row × column

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            category=RuntimeWarning,
        )


        composite = np.nanmedian(
            stack,
            axis=0,
        )


    return (
        composite,
        raster_infos[
            0
        ],
    )


# =============================================================================
# 28. Save biweekly composite
# =============================================================================

def save_biweekly_composite(
    composite,
    reference_info,
    output_file,
):

    output_file = Path(
        output_file
    )


    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    profile = (
        reference_info[
            "profile"
        ]
        .copy()
    )


    profile.update(
        {

            "driver":
                "GTiff",

            "dtype":
                "float32",

            "count":
                int(
                    composite.shape[
                        0
                    ]
                ),

            "compress":
                "deflate",

            "nodata":
                np.nan,

        }
    )


    with rasterio.open(
        output_file,
        "w",
        **profile,
    ) as dst:

        dst.write(
            composite.astype(
                "float32"
            )
        )


# =============================================================================
# 29. Invalid backup path
# =============================================================================

def get_invalid_backup_path(
    file_path,
):

    file_path = Path(
        file_path
    )


    candidate = (
        file_path
        .with_name(
            file_path.name
            +
            ".invalid"
        )
    )


    counter = 1


    while candidate.exists():

        candidate = (
            file_path
            .with_name(
                file_path.name
                +
                f".invalid_{counter}"
            )
        )


        counter += 1


    return candidate


# =============================================================================
# 30. Quality label
# =============================================================================

def quality_label(
    valid_fraction,
):

    if pd.isna(
        valid_fraction
    ):

        return "missing"


    if valid_fraction >= 0.80:

        return "excellent"


    if valid_fraction >= 0.50:

        return "usable"


    if valid_fraction >= 0.20:

        return "limited"


    if valid_fraction > 0:

        return "poor"


    return "unusable"


# =============================================================================
# 31. Identify source quality variable
# =============================================================================

def identify_source_quality_column(
    dataframe,
):

    candidates = [

        "valid_pixel_fraction",

        "valid_pixel_fraction_any_band",

        "valid_pixel_fraction_all_bands",

    ]


    for column in candidates:

        if column in dataframe.columns:

            return column


    return None


SOURCE_QUALITY_COLUMN = (
    identify_source_quality_column(
        inventory
    )
)


print(
    "\nSource image-quality variable:"
)


print(
    SOURCE_QUALITY_COLUMN
)


# =============================================================================
# 32. Build / reuse biweekly image
# =============================================================================

def build_or_reuse_biweekly(
    source_files,
    output_file,
    band_names,
):

    output_file = Path(
        output_file
    )


    # =========================================================================
    # Existing output
    # =========================================================================

    if (
        SKIP_EXISTING_BIWEEKLY
        and
        output_file.exists()
    ):

        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if inspection.get(
            "reusable",
            False
        ):

            print(
                "    Existing biweekly TIFF valid — skipping rebuild."
            )


            inspection[
                "build_action"
            ] = (
                "skipped_existing"
            )


            inspection[
                "composite_created"
            ] = 1


            inspection[
                "file_size_bytes"
            ] = (
                output_file
                .stat()
                .st_size
            )


            return inspection


        print(
            "    Existing biweekly TIFF invalid:"
        )


        print(
            "   ",
            inspection.get(
                "validation_status"
            ),
            "|",
            inspection.get(
                "error"
            ),
        )


        invalid_backup = (
            get_invalid_backup_path(
                output_file
            )
        )


        try:

            output_file.rename(
                invalid_backup
            )


            print(
                "    Invalid TIFF moved to:"
            )


            print(
                "   ",
                invalid_backup
            )


        except Exception:

            try:

                output_file.unlink()

            except Exception:

                pass


    # =========================================================================
    # Build output
    # =========================================================================

    try:

        composite, reference_info = (
            create_biweekly_composite(
                source_files
            )
        )


        save_biweekly_composite(

            composite=
                composite,

            reference_info=
                reference_info,

            output_file=
                output_file,

        )


        inspection = (
            inspect_tiff(
                output_file,
                band_names,
            )
        )


        if not inspection.get(
            "reusable",
            False
        ):

            return {

                "composite_created":
                    0,

                "build_action":
                    "failed",

                "valid_pixel_fraction":
                    np.nan,

                "valid_pixel_percentage":
                    np.nan,

                "error":
                    inspection.get(
                        "error"
                    ),

            }


        inspection[
            "composite_created"
        ] = 1


        inspection[
            "build_action"
        ] = (
            "created"
        )


        inspection[
            "file_size_bytes"
        ] = (
            output_file
            .stat()
            .st_size
        )


        return inspection


    except Exception as error:

        return {

            "composite_created":
                0,

            "build_action":
                "failed",

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "error":
                str(
                    error
                ),

        }


# =============================================================================
# 33. Build fixed site × sensor master
#
# IMPORTANT:
#
# selected_site_sample.csv defines the sample.
#
# We do NOT derive the panel from acquisitions.
# =============================================================================

sensor_master = pd.DataFrame(
    {

        "sensor": [

            "sentinel1",

            "sentinel2",

        ]

    }
)


selected_sample[
    "_merge_key"
] = 1


sensor_master[
    "_merge_key"
] = 1


site_sensor_master = (
    selected_sample
    .merge(
        sensor_master,
        on=
            "_merge_key",
    )
    .drop(
        columns=
            "_merge_key"
    )
)


selected_sample = (
    selected_sample
    .drop(
        columns=
            "_merge_key"
    )
)


print(
    "\nSite × sensor combinations:"
)


print(
    len(
        site_sensor_master
    )
)


print(
    "\nExpected site × sensor combinations:"
)


print(
    EXPECTED_TOTAL_SITES
    *
    2
)


# =============================================================================
# 34. Optional test mode
# =============================================================================

if RUN_TEST_ONLY:

    first_treatment = (
        selected_sample
        .loc[
            selected_sample[
                "group"
            ]
            ==
            "treatment"
        ]
        .iloc[
            0
        ][
            "site_id"
        ]
    )


    first_control = (
        selected_sample
        .loc[
            selected_sample[
                "group"
            ]
            ==
            "counterfactual"
        ]
        .iloc[
            0
        ][
            "site_id"
        ]
    )


    selected_test_ids = [

        str(
            first_treatment
        ),

        str(
            first_control
        ),

    ]


    site_sensor_to_process = (
        site_sensor_master
        .loc[
            site_sensor_master[
                "site_id"
            ]
            .astype(str)
            .isin(
                selected_test_ids
            )
        ]
        .copy()
    )


    print(
        "\nTEST MODE ACTIVE"
    )


    print(
        "Selected sites:"
    )


    print(
        selected_test_ids
    )


else:

    site_sensor_to_process = (
        site_sensor_master
        .copy()
    )


    print(
        "\nFULL DATASET MODE"
    )


# =============================================================================
# 35. Generate biweekly composites
# =============================================================================

biweekly_records = []


total_combinations = len(
    site_sensor_to_process
)


for combination_number, (_, combination) in enumerate(
    site_sensor_to_process.iterrows(),
    start=1,
):

    site_id = str(
        combination[
            "site_id"
        ]
    )


    group = str(
        combination[
            "group"
        ]
    )


    sensor = str(
        combination[
            "sensor"
        ]
    )


    matched_treatment_site_id = (
        combination[
            "matched_treatment_site_id"
        ]
    )


    control_rank = (
        combination[
            "control_rank"
        ]
    )


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{combination_number}/{total_combinations}"
    )


    print(
        site_id,
        "|",
        group,
        "|",
        sensor,
    )


    if group == "counterfactual":

        print(
            "Matched treatment:",
            matched_treatment_site_id,
            "| control rank:",
            control_rank,
        )


    print(
        "=" * 100
    )


    # -------------------------------------------------------------------------
    # Acquisition records for this fixed site × sensor.
    #
    # This may legitimately be empty.
    # -------------------------------------------------------------------------

    site_inventory = (
        inventory
        .loc[
            (
                inventory[
                    "site_id"
                ]
                ==
                site_id
            )
            &
            (
                inventory[
                    "group"
                ]
                ==
                group
            )
            &
            (
                inventory[
                    "sensor"
                ]
                ==
                sensor
            )
        ]
        .copy()
    )


    # -------------------------------------------------------------------------
    # Output sensor root
    # -------------------------------------------------------------------------

    if sensor == "sentinel1":

        sensor_root = (
            S1_BIWEEKLY_DIR
        )


    elif sensor == "sentinel2":

        sensor_root = (
            S2_BIWEEKLY_DIR
        )


    else:

        print(
            "Unknown sensor — skipping:"
        )


        print(
            sensor
        )


        continue


    band_names = (
        SENSOR_BANDS[
            sensor
        ]
    )


    # =========================================================================
    # Every fixed 14-day period
    # =========================================================================

    for _, period_row in (
        period_definitions.iterrows()
    ):

        period = str(
            period_row[
                "period"
            ]
        )


        period_number = int(
            period_row[
                "period_number"
            ]
        )


        period_id = str(
            period_row[
                "period_id"
            ]
        )


        period_start = pd.Timestamp(
            period_row[
                "period_start"
            ]
        )


        period_end = pd.Timestamp(
            period_row[
                "period_end"
            ]
        )


        # ---------------------------------------------------------------------
        # Source acquisitions inside this 14-day period
        # -------------------------------------------------------------------------

        acquisitions = (
            site_inventory
            .loc[
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    >=
                    period_start
                )
                &
                (
                    site_inventory[
                        "acquisition_date"
                    ]
                    <=
                    period_end
                )
            ]
            .sort_values(
                "acquisition_date"
            )
            .copy()
        )


        acquisition_count = (
            len(
                acquisitions
            )
        )


        acquisition_dates = (
            acquisitions[
                "acquisition_date"
            ]
            .dt.strftime(
                "%Y-%m-%d"
            )
            .tolist()
        )


        source_files = (
            acquisitions[
                "file_path"
            ]
            .astype(str)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # Source-image quality
        # -------------------------------------------------------------------------

        source_mean_valid_fraction = (
            np.nan
        )


        source_best_valid_fraction = (
            np.nan
        )


        if (
            acquisition_count > 0
            and
            SOURCE_QUALITY_COLUMN is not None
        ):

            quality_values = pd.to_numeric(
                acquisitions[
                    SOURCE_QUALITY_COLUMN
                ],
                errors=
                    "coerce",
            ).dropna()


            if len(
                quality_values
            ) > 0:

                source_mean_valid_fraction = float(
                    quality_values.mean()
                )


                source_best_valid_fraction = float(
                    quality_values.max()
                )


        # ---------------------------------------------------------------------
        # Base panel row
        # -------------------------------------------------------------------------

        record = {

            "site_id":
                site_id,

            "group":
                group,

            "matched_treatment_site_id":
                matched_treatment_site_id,

            "control_rank":
                control_rank,

            "sensor":
                sensor,

            "period":
                period,

            "period_number":
                period_number,

            "period_id":
                period_id,

            "period_start":
                period_start.strftime(
                    "%Y-%m-%d"
                ),

            "period_end":
                period_end.strftime(
                    "%Y-%m-%d"
                ),

            "calendar_days":
                14,

            "acquisition_count":
                acquisition_count,

            "acquisition_dates":
                json.dumps(
                    acquisition_dates
                ),

            "source_files":
                json.dumps(
                    source_files
                ),

            "has_data":
                int(
                    acquisition_count > 0
                ),

            "multiple_acquisitions":
                int(
                    acquisition_count > 1
                ),

            "source_mean_valid_pixel_fraction":
                source_mean_valid_fraction,

            "source_best_valid_pixel_fraction":
                source_best_valid_fraction,

            "composite_created":
                0,

            "build_action":
                "missing",

            "output_file":
                None,

            "file_size_bytes":
                np.nan,

            "valid_pixel_fraction":
                np.nan,

            "valid_pixel_percentage":
                np.nan,

            "valid_pixel_fraction_any_band":
                np.nan,

            "valid_pixel_percentage_any_band":
                np.nan,

            "valid_pixel_fraction_all_bands":
                np.nan,

            "valid_pixel_percentage_all_bands":
                np.nan,

            "quality_label":
                "missing",

            "improvement_vs_mean_source":
                np.nan,

            "improvement_vs_best_source":
                np.nan,

            "error":
                None,

        }


        print(
            "\n",
            period_id,
            " | ",
            period_start.strftime(
                "%Y-%m-%d"
            ),
            " to ",
            period_end.strftime(
                "%Y-%m-%d"
            ),
            " | acquisitions = ",
            acquisition_count,
            sep=""
        )


        # ---------------------------------------------------------------------
        # Missing period
        # -------------------------------------------------------------------------

        if acquisition_count == 0:

            print(
                "    No acquisition in this 14-day period."
            )


            biweekly_records.append(
                record
            )


            continue


        # ---------------------------------------------------------------------
        # Output filename
        # -------------------------------------------------------------------------

        output_file = (

            sensor_root /
            group /
            period /
            (
                f"{site_id}_"
                f"{period_id}_"
                f"{period_start.strftime('%Y-%m-%d')}_"
                f"{period_end.strftime('%Y-%m-%d')}_"
                f"biweekly_"
                f"{sensor}.tif"
            )

        )


        # ---------------------------------------------------------------------
        # Create/reuse composite
        # -------------------------------------------------------------------------

        result = (
            build_or_reuse_biweekly(

                source_files=
                    source_files,

                output_file=
                    output_file,

                band_names=
                    band_names,

            )
        )


        record.update(
            result
        )


        if (
            result.get(
                "composite_created",
                0
            )
            ==
            1
        ):

            record[
                "output_file"
            ] = (
                str(
                    output_file
                )
            )


        # ---------------------------------------------------------------------
        # Quality and improvement
        # -------------------------------------------------------------------------

        if pd.notna(
            record.get(
                "valid_pixel_fraction"
            )
        ):

            current_quality = float(
                record[
                    "valid_pixel_fraction"
                ]
            )


            record[
                "quality_label"
            ] = (
                quality_label(
                    current_quality
                )
            )


            if pd.notna(
                source_mean_valid_fraction
            ):

                record[
                    "improvement_vs_mean_source"
                ] = (
                    current_quality
                    -
                    source_mean_valid_fraction
                )


            if pd.notna(
                source_best_valid_fraction
            ):

                record[
                    "improvement_vs_best_source"
                ] = (
                    current_quality
                    -
                    source_best_valid_fraction
                )


            print(
                "    Valid pixels:",
                round(
                    record[
                        "valid_pixel_percentage"
                    ],
                    2,
                ),
                "%"
            )


            print(
                "    Quality:",
                record[
                    "quality_label"
                ]
            )


            print(
                "    Action:",
                record[
                    "build_action"
                ]
            )


        else:

            print(
                "    Composite unavailable."
            )


        biweekly_records.append(
            record
        )


# =============================================================================
# 36. Build detailed quality dataset
# =============================================================================

biweekly_quality = pd.DataFrame(
    biweekly_records
)


if biweekly_quality.empty:

    raise RuntimeError(
        "No biweekly panel rows were generated."
    )


# =============================================================================
# 37. Quality thresholds
#
# Missing images remain zero in threshold indicators,
# but percentages below use periods_with_data as denominator.
# =============================================================================

for threshold in [

    0.40,

    0.50,

    0.60,

    0.80,

    0.90,

]:

    threshold_pct = int(
        threshold
        *
        100
    )


    biweekly_quality[
        f"quality_ge_{threshold_pct}pct"
    ] = (
        biweekly_quality[
            "valid_pixel_fraction"
        ]
        >=
        threshold
    ).astype(int)


biweekly_quality[
    "quality_100pct"
] = (
    biweekly_quality[
        "valid_pixel_fraction"
    ]
    >=
    0.999999
).astype(int)


# =============================================================================
# 38. Save detailed biweekly quality
# =============================================================================

biweekly_quality.to_csv(
    QUALITY_CSV_FILE,
    index=False,
)


print(
    "\nBiweekly image-quality CSV:"
)


print(
    QUALITY_CSV_FILE
)


# =============================================================================
# 39. Build/reuse summary
# =============================================================================

build_action_summary = (
    biweekly_quality[
        "build_action"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "build_action"
    )
    .reset_index(
        name=
            "image_count"
    )
)


build_action_summary.to_csv(
    BUILD_ACTION_FILE,
    index=False,
)


print(
    "\nBuild/reuse summary:"
)


print(
    build_action_summary.to_string(
        index=False
    )
)


# =============================================================================
# 40. Overall quality summary
# =============================================================================

biweekly_summary = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        expected_site_periods=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        periods_with_multiple_acquisitions=(
            "multiple_acquisitions",
            "sum",
        ),

        total_acquisitions=(
            "acquisition_count",
            "sum",
        ),

        mean_acquisitions_per_period=(
            "acquisition_count",
            "mean",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        periods_ge_40pct_valid=(
            "quality_ge_40pct",
            "sum",
        ),

        periods_ge_50pct_valid=(
            "quality_ge_50pct",
            "sum",
        ),

        periods_ge_60pct_valid=(
            "quality_ge_60pct",
            "sum",
        ),

        periods_ge_80pct_valid=(
            "quality_ge_80pct",
            "sum",
        ),

        periods_ge_90pct_valid=(
            "quality_ge_90pct",
            "sum",
        ),

        mean_improvement_vs_mean_source=(
            "improvement_vs_mean_source",
            "mean",
        ),

        mean_improvement_vs_best_source=(
            "improvement_vs_best_source",
            "mean",
        ),

    )
)


# =============================================================================
# 41. Availability statistics
# =============================================================================

biweekly_summary[
    "periods_without_data"
] = (
    biweekly_summary[
        "expected_site_periods"
    ]
    -
    biweekly_summary[
        "periods_with_data"
    ]
)


biweekly_summary[
    "percent_periods_with_data"
] = (
    biweekly_summary[
        "periods_with_data"
    ]
    /
    biweekly_summary[
        "expected_site_periods"
    ]
    *
    100
)


# =============================================================================
# 42. Threshold statistics
# =============================================================================

for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    biweekly_summary[
        f"percent_available_periods_ge_{threshold}pct"
    ] = np.where(

        biweekly_summary[
            "periods_with_data"
        ]
        >
        0,

        biweekly_summary[
            f"periods_ge_{threshold}pct_valid"
        ]
        /
        biweekly_summary[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


# =============================================================================
# 43. Convert fractions to percentages
# =============================================================================

for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    biweekly_summary[
        target_column
    ] = (
        biweekly_summary[
            source_column
        ]
        *
        100
    )


biweekly_summary[
    "mean_improvement_vs_mean_source_percentage_points"
] = (
    biweekly_summary[
        "mean_improvement_vs_mean_source"
    ]
    *
    100
)


biweekly_summary[
    "mean_improvement_vs_best_source_percentage_points"
] = (
    biweekly_summary[
        "mean_improvement_vs_best_source"
    ]
    *
    100
)


biweekly_summary.to_csv(
    SUMMARY_CSV_FILE,
    index=False,
)


# =============================================================================
# 44. PERIOD DIMENSION
#
# Across sites within each 14-day period.
# =============================================================================

period_dimension = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "period",
            "period_number",
            "period_id",
            "period_start",
            "period_end",
        ],
        as_index=False,
    )
    .agg(

        number_of_sites=(
            "site_id",
            "nunique",
        ),

        images_with_data=(
            "has_data",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    period_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        period_dimension[
            "images_with_data"
        ]
        >
        0,

        period_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        period_dimension[
            "images_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    period_dimension[
        target_column
    ] = (
        period_dimension[
            source_column
        ]
        *
        100
    )


period_dimension.to_csv(
    PERIOD_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 45. SITE DIMENSION
#
# For each individual spatial unit, evaluate quality across time.
#
# Matching information is retained.
# =============================================================================

site_dimension = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
            "site_id",
            "matched_treatment_site_id",
            "control_rank",
            "period",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(

        expected_periods=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        mean_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "mean",
        ),

        median_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "median",
        ),

        min_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "min",
        ),

        max_valid_pixel_fraction=(
            "valid_pixel_fraction",
            "max",
        ),

        images_ge_40pct=(
            "quality_ge_40pct",
            "sum",
        ),

        images_ge_50pct=(
            "quality_ge_50pct",
            "sum",
        ),

        images_ge_60pct=(
            "quality_ge_60pct",
            "sum",
        ),

        images_ge_80pct=(
            "quality_ge_80pct",
            "sum",
        ),

        images_ge_90pct=(
            "quality_ge_90pct",
            "sum",
        ),

    )
)


site_dimension[
    "periods_without_data"
] = (
    site_dimension[
        "expected_periods"
    ]
    -
    site_dimension[
        "periods_with_data"
    ]
)


site_dimension[
    "availability_percentage"
] = (
    site_dimension[
        "periods_with_data"
    ]
    /
    site_dimension[
        "expected_periods"
    ]
    *
    100
)


for threshold in [

    40,

    50,

    60,

    80,

    90,

]:

    site_dimension[
        f"percent_images_ge_{threshold}pct"
    ] = np.where(

        site_dimension[
            "periods_with_data"
        ]
        >
        0,

        site_dimension[
            f"images_ge_{threshold}pct"
        ]
        /
        site_dimension[
            "periods_with_data"
        ]
        *
        100,

        np.nan,

    )


for source_column, target_column in [

    (
        "mean_valid_pixel_fraction",
        "mean_valid_pixel_percentage",
    ),

    (
        "median_valid_pixel_fraction",
        "median_valid_pixel_percentage",
    ),

    (
        "min_valid_pixel_fraction",
        "min_valid_pixel_percentage",
    ),

    (
        "max_valid_pixel_fraction",
        "max_valid_pixel_percentage",
    ),

]:

    site_dimension[
        target_column
    ] = (
        site_dimension[
            source_column
        ]
        *
        100
    )


site_dimension.to_csv(
    SITE_DIMENSION_FILE,
    index=False,
)


# =============================================================================
# 46. Quality definitions
# =============================================================================

quality_definitions = pd.DataFrame(
    {

        "variable": [

            "site_id",

            "matched_treatment_site_id",

            "control_rank",

            "period_id",

            "acquisition_count",

            "has_data",

            "multiple_acquisitions",

            "valid_pixel_fraction",

            "valid_pixel_fraction_any_band",

            "valid_pixel_fraction_all_bands",

            "source_mean_valid_pixel_fraction",

            "source_best_valid_pixel_fraction",

            "improvement_vs_mean_source",

            "improvement_vs_best_source",

            "build_action",

        ],


        "meaning": [

            (
                "Treatment or counterfactual spatial-unit ID."
            ),

            (
                "Treatment site associated with the spatial unit. "
                "Treatment rows map to themselves."
            ),

            (
                "Counterfactual rank within the matched treatment set."
            ),

            (
                "Fixed 14-day period identifier, e.g. before_P01."
            ),

            (
                "Number of actual Notebook 10 acquisitions contributing "
                "to the 14-day composite."
            ),

            (
                "1 if at least one acquisition exists in the 14-day period; "
                "0 otherwise."
            ),

            (
                "1 if two or more acquisitions contribute to the composite."
            ),

            (
                "PRIMARY quality metric: fraction of spatial pixels where "
                "at least one output band contains a finite value."
            ),

            (
                "Explicit any-band version of the primary quality metric."
            ),

            (
                "Strict diagnostic requiring every output band to be valid "
                "at the spatial pixel."
            ),

            (
                "Average primary valid-pixel fraction across source "
                "acquisition-level images."
            ),

            (
                "Highest primary valid-pixel fraction among source "
                "acquisition-level images."
            ),

            (
                "Biweekly quality minus average source-image quality."
            ),

            (
                "Biweekly quality minus best source-image quality."
            ),

            (
                "created = new TIFF; skipped_existing = valid TIFF already "
                "exists; missing = no acquisition; failed = processing error."
            ),

        ],

    }
)


# =============================================================================
# 47. Excel workbook
# =============================================================================

try:

    with pd.ExcelWriter(
        QUALITY_EXCEL_FILE,
        engine=
            "openpyxl",
    ) as writer:

        biweekly_quality.to_excel(
            writer,
            sheet_name=
                "image_quality",
            index=False,
        )


        biweekly_summary.to_excel(
            writer,
            sheet_name=
                "summary",
            index=False,
        )


        period_dimension.to_excel(
            writer,
            sheet_name=
                "period_dimension",
            index=False,
        )


        site_dimension.to_excel(
            writer,
            sheet_name=
                "site_dimension",
            index=False,
        )


        period_definitions.to_excel(
            writer,
            sheet_name=
                "period_definitions",
            index=False,
        )


        selected_sample.to_excel(
            writer,
            sheet_name=
                "selected_sample",
            index=False,
        )


        sample_validation.to_excel(
            writer,
            sheet_name=
                "sample_validation",
            index=False,
        )


        quality_definitions.to_excel(
            writer,
            sheet_name=
                "quality_definitions",
            index=False,
        )


        build_action_summary.to_excel(
            writer,
            sheet_name=
                "build_actions",
            index=False,
        )


    print(
        "\nExcel workbook saved:"
    )


    print(
        QUALITY_EXCEL_FILE
    )


except ModuleNotFoundError:

    print(
        "\nopenpyxl is not installed."
    )


    print(
        "CSV outputs were still generated."
    )


    print(
        "Install with:"
    )


    print(
        "%pip install openpyxl"
    )


# =============================================================================
# 48. Folder summary
# =============================================================================

folder_records = []


for sensor_name, sensor_root in [

    (
        "sentinel1",
        S1_BIWEEKLY_DIR,
    ),

    (
        "sentinel2",
        S2_BIWEEKLY_DIR,
    ),

]:

    for group in [

        "treatment",

        "counterfactual",

    ]:

        for period in [

            "before",

            "after",

        ]:

            folder = (

                sensor_root /
                group /
                period

            )


            folder_records.append(
                {

                    "sensor":
                        sensor_name,

                    "group":
                        group,

                    "period":
                        period,

                    "folder":
                        str(
                            folder
                        ),

                    "biweekly_tiff_count":
                        len(
                            list(
                                folder.glob(
                                    "*.tif"
                                )
                            )
                        ),

                }
            )


folder_summary = pd.DataFrame(
    folder_records
)


folder_summary.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False,
)


# =============================================================================
# 49. Fixed panel validation
# =============================================================================

coverage_check = (
    biweekly_quality
    .groupby(
        [
            "sensor",
            "group",
        ],
        as_index=False,
    )
    .agg(

        unique_sites=(
            "site_id",
            "nunique",
        ),

        total_site_period_rows=(
            "period_id",
            "count",
        ),

        periods_with_data=(
            "has_data",
            "sum",
        ),

        composites_available=(
            "composite_created",
            "sum",
        ),

    )
)


print(
    "\n"
    + "=" * 100
)


print(
    "BIWEEKLY PANEL VALIDATION"
)


print(
    "=" * 100
)


print(
    coverage_check.to_string(
        index=False
    )
)


# =============================================================================
# 50. Expected panel dimensions
# =============================================================================

expected_rows_per_sensor = (
    total_site_count
    *
    EXPECTED_PERIODS_TOTAL
)


expected_rows_both_sensors = (
    expected_rows_per_sensor
    *
    2
)


actual_panel_rows = (
    len(
        biweekly_quality
    )
)


print(
    "\nExpected treatment sites:"
)


print(
    EXPECTED_TREATMENT_SITES
)


print(
    "\nExpected counterfactual sites:"
)


print(
    EXPECTED_COUNTERFACTUAL_SITES
)


print(
    "\nActual selected sites:"
)


print(
    total_site_count
)


print(
    "\nExpected periods per site:"
)


print(
    "10 before + 10 after = 20"
)


print(
    "\nExpected rows per sensor based on actual selected sample:"
)


print(
    expected_rows_per_sensor
)


print(
    "\nExpected rows for both sensors:"
)


print(
    expected_rows_both_sensors
)


print(
    "\nActual biweekly panel rows:"
)


print(
    actual_panel_rows
)


if (
    not RUN_TEST_ONLY
    and
    actual_panel_rows
    !=
    expected_rows_both_sensors
):

    print(
        "\nWARNING:"
    )


    print(
        "Biweekly panel row count does not match expected fixed panel."
    )


# =============================================================================
# 51. Main quality summary
# =============================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "BIWEEKLY IMAGE QUALITY SUMMARY"
)


print(
    "=" * 110
)


columns_to_show = [

    "sensor",

    "group",

    "period",

    "number_of_sites",

    "expected_site_periods",

    "periods_with_data",

    "periods_without_data",

    "percent_periods_with_data",

    "periods_with_multiple_acquisitions",

    "mean_acquisitions_per_period",

    "mean_valid_pixel_percentage",

    "median_valid_pixel_percentage",

    "periods_ge_80pct_valid",

    "percent_available_periods_ge_80pct",

]


print(
    biweekly_summary[
        columns_to_show
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 52. Sentinel-2 period quality
# =============================================================================

s2_period = (
    period_dimension
    .loc[
        period_dimension[
            "sensor"
        ]
        ==
        "sentinel2"
    ]
    .copy()
)


print(
    "\n"
    + "=" * 110
)


print(
    "SENTINEL-2 BIWEEKLY PERIOD QUALITY"
)


print(
    "=" * 110
)


print(
    s2_period[
        [
            "group",
            "period",
            "period_id",
            "number_of_sites",
            "images_with_data",
            "mean_valid_pixel_percentage",
            "median_valid_pixel_percentage",
            "images_ge_80pct",
            "percent_images_ge_80pct",
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# 53. Temporal validation
# =============================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "TEMPORAL VALIDATION"
)


print(
    "=" * 110
)


print(
    "\nBefore periods:",
    len(
        before_definition
    )
)


print(
    "After periods:",
    len(
        after_definition
    )
)


print(
    "\nFirst before:"
)


print(
    before_definition.iloc[
        0
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast before:"
)


print(
    before_definition.iloc[
        -1
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nFirst after:"
)


print(
    after_definition.iloc[
        0
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


print(
    "\nLast after:"
)


print(
    after_definition.iloc[
        -1
    ][
        [
            "period_id",
            "period_start",
            "period_end",
        ]
    ]
)


# =============================================================================
# 54. Final summary
# =============================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NOTEBOOK 12 COMPLETE"
)


print(
    "=" * 110
)


print(
    "\nSource acquisition-level dataset:"
)


print(
    DAILY_DIR
)


print(
    "\nBiweekly output:"
)


print(
    BIWEEKLY_DIR
)


print(
    "\nSelected spatial sample:"
)


print(
    treatment_count,
    "treatment sites"
)


print(
    counterfactual_count,
    "counterfactual sites"
)


print(
    total_site_count,
    "total sites"
)


print(
    "\nTemporal design:"
)


print(
    "10 BEFORE × 14 days"
)


print(
    "10 AFTER  × 14 days"
)


print(
    "\nBefore:"
)


print(
    "2024-05-10 through 2024-09-26"
)


print(
    "\nAfter:"
)


print(
    "2024-09-27 through 2025-02-13"
)


print(
    "\nPotential panel rows:"
)


print(
    "Per sensor:",
    expected_rows_per_sensor
)


print(
    "Both sensors:",
    expected_rows_both_sensors
)


print(
    "\nActual panel rows:"
)


print(
    actual_panel_rows
)


print(
    "\nBiweekly image-quality CSV:"
)


print(
    QUALITY_CSV_FILE
)


print(
    "\nBiweekly quality Excel:"
)


print(
    QUALITY_EXCEL_FILE
)


print(
    "\nSummary:"
)


print(
    SUMMARY_CSV_FILE
)


print(
    "\nPeriod dimension:"
)


print(
    PERIOD_DIMENSION_FILE
)


print(
    "\nSite dimension:"
)


print(
    SITE_DIMENSION_FILE
)


print(
    "\nPeriod definitions:"
)


print(
    PERIOD_DEFINITION_FILE
)


print(
    "\nSample validation:"
)


print(
    SAMPLE_VALIDATION_FILE
)


print(
    "\nFolder summary:"
)


print(
    FOLDER_SUMMARY_FILE
)


print(
    "\nBuild/reuse summary:"
)


print(
    BUILD_ACTION_FILE
)


print(
    "\nSkip existing valid biweekly TIFFs:"
)


print(
    SKIP_EXISTING_BIWEEKLY
)


if RUN_TEST_ONLY:

    print(
        "\nTEST MODE ACTIVE."
    )


else:

    print(
        "\nFULL DATASET MODE ACTIVE."
    )


    print(
        "The fixed sample from selected_site_sample.csv was processed."
    )


    print(
        "Sites without imagery remain explicit missing panel rows."
    )


print(
    "\nNo Earth Engine request or download occurs in Notebook 12."
)


print(
    "\nNotebook completed successfully."
)

Packages loaded successfully.

Notebook 10 daily inventory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/daily_satellite_inventory.csv

Notebook 10 selected sample:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets/selected_site_sample.csv

Study period:
2024-05-10 through 2025-02-13

Before:
2024-05-10 through 2024-09-26

After:
2024-09-27 through 2025-02-13

Biweekly output directory:
/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/biweekly_datasets

SELECTED SAMPLE

Treatment sites: 10
Counterfactual sites: 50
Total sites: 60

Controls per treatment:
matched_treatment_site_id  counterfactual_count  expected_counterfactual_count  sample_complete
           treatment_0001                     5                              5             True
           treatment_0002                     5                              5             True
           treatment_0003            